requirements

In particular, we will represent arbitrary (Unicode) strings as a sequence of bytes and train our BPE tokenizer on this byte sequence. 

Later, we will use this
tokenizer to encode text (a string) into tokens (a sequence of integers) for language modeling.

### Problem 1 -- Understanding Unicode
具体要求如下

(a) **What Unicode character does chr(0) return?**

Deliverable: A one-sentence response.

(b) **How does this character’s string representation `(__repr__())` differ from its printed representa-tion?**

Deliverable: A one-sentence response.

(c) **What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:**

```shell
>>> chr(0)
>>> print(chr(0))
>>> "this is a test" + chr(0) + "string"
>>> print("this is a test" + chr(0) + "string")
```

Deliverable: A one-sentence response.

In [5]:
chr(0)

'\x00'

In [6]:
print(chr(0))

 


In [7]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [9]:
print("this is a test" + chr(0) + "string")
print("this is a test" + "string")

this is a test string
this is a teststring


so basically, chr(0) might be a space ?

---

In [15]:
def decode_utf8_bytes_to_str(bytestring: bytes):
    return bytestring.decode("utf-8")


# 错误在 utf-8 的多字节编码的特性
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


to_decode = "你好呀！"
print("this is the encode of all the char: ", to_decode.encode("utf-8"))
print("现在来针对每个字符进行编码: ")
for c in to_decode:
    print(f'"{c}": {c.encode("utf-8")}')


this is the encode of all the char:  b'\xe4\xbd\xa0\xe5\xa5\xbd\xe5\x91\x80\xef\xbc\x81'
现在来针对每个字符进行编码: 
"你": b'\xe4\xbd\xa0'
"好": b'\xe5\xa5\xbd'
"呀": b'\xe5\x91\x80'
"！": b'\xef\xbc\x81'


可以看到对于中文汉字来说，一个汉字的字符就是 e4 bd a0 这三个 bytes,他们是不可分割的一体。

错误的 `def decode_utf8_bytes_to_str_wrong(bytestring: bytes):` 错就错在，单拎出俩一个字节进行解码，这是不符合 utf-8 的编码形式的。

正确的做法应该是直接 decode，因为 utf-8 的编码的好处就是可以在字节流中一下子就区分哪些是同一个字符的字节编码

---

In [17]:
import regex as re
from pprint import pprint

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
# this is a regex-based pre-tokenizer used by GPT-2
pprint(re.findall(PAT, "Some text I'll pre-tokenize using regex."))

['Some', ' text', ' I', "'ll", ' pre', '-', 'tokenize', ' using', ' regex', '.']


In [1]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8")  # raw bytes
tokens = list(
    map(int, tokens)
)  # convert to a list of integers in range 0..255 for convenience
print("---")
print(text)
print("length:", len(text))
print("---")
print(tokens)
print("length:", len(tokens))


---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

In [2]:
print(type(tokens))

<class 'list'>


In [8]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


cnt: dict[tuple[int, int], int] = get_stats(tokens)
top_pair = max(cnt, key=lambda k: cnt[k])

In [ ]:
"""
def merge(ids, pair: tuple[int, int], idx):
    # in the list of ints (ids), replace all constructive occurances of pair with the
    # new token idx
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids
"""

我现在需要设计的函数有
1. 从原始文本中获取原始 raw bytes 流 -- `get_stats`
2. 开始 merge文本，扩充词汇表 -- `merge`
3. 对输入文本进行编码 -- `encode`
4. 解码 -- `decode`

## Python 中的 `bytes` 类型

`bytes` 是 Python 中用于表示**不可变的字节序列**的数据类型，每个元素是 0~255 之间的整数。它是处理二进制数据的核心类型。

---

### 与 `str` 的核心区别

| | `str` | `bytes` |
|---|---|---|
| 内容 | Unicode 字符 | 原始字节（0~255）|
| 字面量 | `"hello"` | `b"hello"` |
| 可变版本 | ❌ | `bytearray` |
| 编码相关 | 无需编码 | 需要指定编码 |

---

### 创建方式

```python
# 1. 字面量
b1 = b"hello"

# 2. 从字符串编码
b2 = "你好".encode("utf-8")       # b'\xe4\xbd\xa0\xe5\xa5\xbd'

# 3. 从整数列表
b3 = bytes([72, 101, 108, 108, 111])  # b'Hello'

# 4. 指定长度的零字节
b4 = bytes(5)                     # b'\x00\x00\x00\x00\x00'
```

---

### 常见操作

```python
data = b"Hello, World!"

# 索引 → 返回整数
print(data[0])        # 72（不是 'H'）

# 切片 → 返回 bytes
print(data[0:5])      # b'Hello'

# 长度
print(len(data))      # 13

# 解码回字符串
print(data.decode("utf-8"))  # Hello, World!

# 查找
print(data.find(b"World"))   # 7

# 拼接
combined = b"foo" + b"bar"   # b'foobar'

# 十六进制表示
print(data.hex())     # '48656c6c6f2c20576f726c6421'
print(bytes.fromhex("48656c6c6f"))  # b'Hello'
```
---

`bytes` 和 `str` 的相互转换，必须明确指定编码 -- 我使用 `utf-8`

In [ ]:
def get_stats(ids: list[int]):
    counts = {}
    # 这里使用 zip 将 ids ids[1:] -- 即前后相邻的 raw bytes 构造成一个 pair
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids: list[int], pair: tuple[int, int], idx: int):
    """用于将原始文本流，根据上一轮迭代获得的出现频率最高的 pair 进行替换压缩
        出现频率最高的 pair 在词汇表中对应的 token id 就是 idx
    Args:
        ids (list[int]): 原始文本流(已经根据初始的词汇表编码成 list[int])
        pair (tuple[int, int]): 上一轮迭代中出现频率最高的 pair
        idx (int): 出现频率最高的 pair 在词汇表中对应的 id
    """
    new_ids: list[int] = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


def encode(text: str):
    tokens = list(map(int, text.encode("utf-8")))
    while len(tokens) >= 2:
        ids = get_stats(tokens)
        to_merge = min(ids, key=lambda )